## Andmete allalaadimine

Selles märkmikus laadime alla Äriregistri avaandmed.
Skript tõmbab alla vajalikud ZIP-failid, mis sisaldavad ettevõtete põhiandmeid ja majandusaasta aruannete detaile (2019-2025), ning pakib need lahti `raw_data` kausta edasiseks töötlemiseks.

In [ ]:
# Vajalike teekide importimine failide allalaadimiseks ja töötlemiseks
import os
import zipfile
from pathlib import Path
from urllib.parse import urlparse
import requests

# Äriregistri avaandmete failide URL-id
zip_urls = [
    "https://avaandmed.ariregister.rik.ee/sites/default/files/avaandmed/ettevotja_rekvisiidid__lihtandmed.csv.zip",
    "https://avaandmed.ariregister.rik.ee/sites/default/files/1.aruannete_yldandmed_kuni_31032026.zip",
    "https://avaandmed.ariregister.rik.ee/sites/default/files/2.EMTAK_myygitulu_kuni_31032026.zip",
    "https://avaandmed.ariregister.rik.ee/sites/default/files/3.myygitulu_geograafiline_kuni_31032026.zip",
    "https://avaandmed.ariregister.rik.ee/sites/default/files/4.2019_aruannete_elemendid_kuni_31032026.zip",
    "https://avaandmed.ariregister.rik.ee/sites/default/files/4.2020_aruannete_elemendid_kuni_31032026.zip",
    "https://avaandmed.ariregister.rik.ee/sites/default/files/4.2021_aruannete_elemendid_kuni_31032026.zip",
    "https://avaandmed.ariregister.rik.ee/sites/default/files/4.2022_aruannete_elemendid_kuni_31032026.zip",
    "https://avaandmed.ariregister.rik.ee/sites/default/files/4.2023_aruannete_elemendid_kuni_31032026.zip",
    "https://avaandmed.ariregister.rik.ee/sites/default/files/4.2024_aruannete_elemendid_kuni_31032026.zip",
    "https://avaandmed.ariregister.rik.ee/sites/default/files/4.2025_aruannete_elemendid_kuni_31032026.zip"
]

In [ ]:
# Loome kausta toorandmete salvestamiseks, kui seda veel ei eksisteeri
raw_data_dir = Path("raw_data")
raw_data_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Skript andmefailide allalaadimiseks ja lahtipakkimiseks
for i, url in enumerate(zip_urls, start=1):
    filename = Path(urlparse(url).path).name or f"file_{i}.zip"
    if not filename.endswith(".zip"):
        filename += ".zip"

    zip_path = raw_data_dir / filename
    expected_csv = raw_data_dir / filename.replace(".zip", ".csv")

    # Kontrollime, kas fail on juba alla laaditud ja lahti pakitud
    if expected_csv.exists():
        print(f"CSV fail on juba olemas, jätame vahele: {expected_csv}")
        continue

    print(f"Laen alla {url}...")
    response = requests.get(url, stream=True)
    response.raise_for_status()

    # Salvestame faili tükkide kaupa, et vältida mälu ületäitumist
    with open(zip_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

    # Kontrollime, kas tegemist on korrektse ZIP-failiga
    if not zipfile.is_zipfile(zip_path):
        print(f"Allalaaditud fail ei ole korrektne ZIP-fail: {zip_path}")
        zip_path.unlink(missing_ok=True)
        continue

    print(f"Pakin lahti {zip_path}...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(raw_data_dir)

    print(f"Kustutan ZIP-faili: {zip_path}")
    zip_path.unlink(missing_ok=True)

print("Andmete allalaadimine ja lahtipakkimine on lõpetatud.")

Unpacking raw_data\ettevotja_rekvisiidid__lihtandmed.csv.zip...
Removing zip: raw_data\ettevotja_rekvisiidid__lihtandmed.csv.zip
CSV already exists, skipping: raw_data\1.aruannete_yldandmed_kuni_31032026.csv
CSV already exists, skipping: raw_data\2.EMTAK_myygitulu_kuni_31032026.csv
CSV already exists, skipping: raw_data\3.myygitulu_geograafiline_kuni_31032026.csv
CSV already exists, skipping: raw_data\4.2019_aruannete_elemendid_kuni_31032026.csv
CSV already exists, skipping: raw_data\4.2020_aruannete_elemendid_kuni_31032026.csv
CSV already exists, skipping: raw_data\4.2021_aruannete_elemendid_kuni_31032026.csv
CSV already exists, skipping: raw_data\4.2022_aruannete_elemendid_kuni_31032026.csv
CSV already exists, skipping: raw_data\4.2023_aruannete_elemendid_kuni_31032026.csv
CSV already exists, skipping: raw_data\4.2024_aruannete_elemendid_kuni_31032026.csv
CSV already exists, skipping: raw_data\4.2025_aruannete_elemendid_kuni_31032026.csv
Done.
